In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

outcome_matrix = pd.read_csv("fct_outcome_matrix.csv", index_col=0)
cost_vector    = pd.read_csv("fct_cost_vector.csv", index_col=0, header=0)
cost_vector.columns = ['mean_cost_s']
unit_summary   = pd.read_csv("fct_unit_summary.csv")

C_full_cost = cost_vector['mean_cost_s'].sum()

print(f"Outcome matrix:     {outcome_matrix.shape}")
print(f"Full sequence cost: {C_full_cost:.2f} s")

In [ ]:
all_steps = outcome_matrix.columns.tolist()
print(f"All steps: {len(all_steps)}")

In [ ]:
fail_matrix = outcome_matrix.loc[
    outcome_matrix.index.isin(failing_units)
].copy()

detection_matrix = (fail_matrix == 0).astype(float).fillna(0)

step_detection_count = detection_matrix.sum(axis=0)
diagnostic_steps     = step_detection_count[
    step_detection_count > 0
].index.tolist()

costs = cost_vector.reindex(
    outcome_matrix.columns
)['mean_cost_s'].fillna(0)

print(f"Detection matrix: {detection_matrix.shape}")
print(f"Diagnostic steps: {len(diagnostic_steps)}")
print(f"Non-diagnostic:   {outcome_matrix.shape[1] - len(diagnostic_steps)}")

In [ ]:
detection_sets = {}
for step in diagnostic_steps:
    detection_sets[step] = set(
        detection_matrix.index[
            detection_matrix[step] == 1
        ].tolist()
    )


costs_scalar = {}
for step in outcome_matrix.columns:
    val = costs.loc[step] if step in costs.index else 0
    if hasattr(val, '__len__'):
        val = float(val.iloc[0])
    else:
        val = float(val)
    costs_scalar[step] = val

step_costs = {
    step: max(costs_scalar.get(step, 0.001), 0.001)
    for step in diagnostic_steps
}

def greedy_cover(max_escapes):
    already_used = set()
    selected     = []
    uncovered    = set(failing_units)

    while len(uncovered) > max_escapes:
        best_step  = None
        best_score = -1
        best_newly = set()

        for step in diagnostic_steps:
            if step in already_used:
                continue
            newly = detection_sets[step] & uncovered
            if not newly:
                continue
            score = len(newly) / step_costs[step]
            if score > best_score:
                best_score = score
                best_step  = step
                best_newly = newly

        if best_step is None:
            break

        already_used.add(best_step)
        selected.append(best_step)
        uncovered -= best_newly

    n_escaped   = len(uncovered)
    subset_cost = sum(costs_scalar.get(s, 0.0) for s in selected)
    return selected, n_escaped, float(subset_cost)

print("Greedy cover function defined")

In [ ]:
print(f"\n{'='*55}")
print("SWEEPING ALLOWED ESCAPES 0 → N_FAIL")
print(f"{'='*55}")


sweep_points = list(range(0, N_fail + 1))

pareto_results  = []
prev_saving_pct = None

for max_esc in sweep_points:
    selected, n_escaped, subset_cost = greedy_cover(max_esc)

    escape_risk     = n_escaped / N_fail
    cost_saving_s   = C_full_cost - subset_cost
    cost_saving_pct = round(cost_saving_s / C_full_cost * 100, 4)

    if cost_saving_pct == prev_saving_pct:
        continue
    prev_saving_pct = cost_saving_pct

    pareto_results.append({
        'max_allowed_escapes': max_esc,
        'escaped_units':       n_escaped,
        'escape_risk':         round(escape_risk, 6),
        'escape_dpm':          round(escape_risk * 1_000_000, 2),
        'steps_selected':      len(selected),
        'subset_cost_s':       round(subset_cost, 4),
        'cost_saving_s':       round(float(cost_saving_s), 4),
        'cost_saving_pct':     cost_saving_pct,
        'selected_steps':      selected
    })

pareto_df = pd.DataFrame(pareto_results)
print(f"Pareto points: {len(pareto_df)}")
print(pareto_df[['escape_risk', 'cost_saving_pct', 'steps_selected']].head(10).to_string())

In [63]:
def eval_subset(subset):
    subset_cost = sum(costs.get(s, 0) for s in subset)
    saving_pct  = (C_full_cost - subset_cost) / C_full_cost * 100
    sub_detect  = detection_matrix.reindex(
        columns=[s for s in subset if s in detection_matrix.columns],
        fill_value=0
    )
    escaped     = (sub_detect.sum(axis=1) == 0).sum()
    escape_risk = escaped / N_fail
    return round(saving_pct, 4), round(escape_risk, 6)

In [ ]:
max_escapes_threshold = delta * N_fail
print(f"Quality threshold allows: {max_escapes_threshold:.4f} units")
print(f"Rounded down:             "
      f"{int(np.floor(max_escapes_threshold))} units")
print(f"Effective requirement:    ZERO escapes")

zero_escape = pareto_df[pareto_df['escaped_units'] == 0]

if len(zero_escape) > 0:
    operating_point = zero_escape.sort_values(
        'cost_saving_pct', ascending=False
    ).iloc[0].copy()
    operating_point['escape_dpm'] = 0.0
    print(f"\nOperating point — zero escape solution found")
else:
    operating_point = pareto_df.sort_values(
        'cost_saving_pct', ascending=False
    ).iloc[0].copy()
    print(f"\nWarning — no zero escape solution found")

print(f"\n{'='*55}")
print("OPERATING POINT")
print(f"{'='*55}")
print(f"Escaped units:      {int(operating_point['escaped_units'])}")
print(f"Steps selected:     {int(operating_point['steps_selected'])}")
print(f"Subset cost:        {operating_point['subset_cost_s']:.2f} s")
print(f"Cost saving:        {operating_point['cost_saving_pct']:.2f}%")
print(f"Time saved/unit:    {operating_point['cost_saving_s']:.2f} s")
print(f"Quality constraint: SATISFIED")

In [ ]:
# Load non-diagnostic steps from RQ1 results
non_diag_df = pd.read_csv('fct_rq1_non_diagnostic.csv')
non_diag_set = set(non_diag_df['test_id'].tolist())

all_steps = outcome_matrix.columns.tolist()

print(f"Non-diagnostic steps: {len(non_diag_df)}")
print(f"All steps: {len(all_steps)}")

In [ ]:

np.random.seed(42)
all_points = []

step_fail_rates = detection_matrix.sum(axis=0).sort_values(ascending=True)
remaining = all_steps.copy()
for step_to_remove in step_fail_rates.index:
    if step_to_remove in remaining:
        remaining = [s for s in remaining if s != step_to_remove]
    saving_pct, escape_risk = eval_subset(remaining)
    all_points.append({'cost_saving_pct': saving_pct, 'escape_risk': escape_risk})


remaining2 = all_steps.copy()
for _, row in non_diag_df.sort_values('cost_s', ascending=False).iterrows():
    step = row['test_id']
    if step in remaining2:
        remaining2 = [s for s in remaining2 if s != step]
    saving_pct, escape_risk = eval_subset(remaining2)
    all_points.append({'cost_saving_pct': saving_pct, 'escape_risk': escape_risk})


for _ in range(2000):
    n_steps = np.random.randint(1, len(all_steps))
    subset  = np.random.choice(
        all_steps, size=n_steps, replace=False
    ).tolist()
    saving_pct, escape_risk = eval_subset(subset)
    all_points.append({'cost_saving_pct': saving_pct, 'escape_risk': escape_risk})


for _ in range(1500):
    n_remove = np.random.randint(0, len(non_diag_set) + 1)
    removed  = set(np.random.choice(
        list(non_diag_set), size=n_remove, replace=False
    ))
    subset = [s for s in all_steps if s not in removed]
    saving_pct, escape_risk = eval_subset(subset)
    all_points.append({'cost_saving_pct': saving_pct, 'escape_risk': escape_risk})

for _ in range(1000):
  
    n_diag_keep = np.random.randint(
        1, len(diagnostic_steps) + 1
    )
    diag_subset = np.random.choice(
        diagnostic_steps, size=n_diag_keep, replace=False
    ).tolist()
    subset = diag_subset  
    saving_pct, escape_risk = eval_subset(subset)
    all_points.append({'cost_saving_pct': saving_pct, 'escape_risk': escape_risk})

all_points_df = pd.DataFrame(all_points)
print(f"Total candidate points: {len(all_points_df)}")


dedup        = all_points_df.drop_duplicates(
    subset=['escape_risk', 'cost_saving_pct']
)
pareto_front = get_pareto_frontier(dedup)
print(f"Pareto front points: {len(pareto_front)}")

In [ ]:
# Full Pareto front from heuristic for plotting
def get_pareto_frontier(df):
    df = df.sort_values('cost_saving_pct', ascending=False).reset_index(drop=True)
    pareto   = []
    min_risk = np.inf
    for _, row in df.iterrows():
        if row['escape_risk'] <= min_risk:
            pareto.append(row)
            min_risk = row['escape_risk']
    return pd.DataFrame(pareto).drop_duplicates(
        subset=['escape_risk', 'cost_saving_pct']
    ).sort_values('escape_risk', ascending=True)

dedup        = all_points_df.drop_duplicates(
    subset=['escape_risk', 'cost_saving_pct']
)
pareto_front = get_pareto_frontier(dedup)

# Greedy sweep for zoom panel and log fit
pareto_front_greedy = pareto_df[['escape_risk', 'cost_saving_pct']].copy()
pareto_front_greedy = pareto_front_greedy.sort_values(
    'escape_risk'
).reset_index(drop=True)

print(f"Heuristic Pareto points: {len(pareto_front)}")
print(f"Greedy Pareto points:    {len(pareto_front_greedy)}")

In [ ]:
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

def style_ax(ax):
    ax.xaxis.set_major_locator(plt.MaxNLocator(5))
    ax.yaxis.set_major_locator(plt.MaxNLocator(10))
    ax.tick_params(axis='both', labelsize=11)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.3)

# Log fit
def log_func(x, a, b):
    return a * np.log(x + 1e-6) + b

x_pareto = pareto_front_greedy['escape_risk'].values
y_pareto = pareto_front_greedy['cost_saving_pct'].values

mask     = x_pareto > 0.001
x_fit    = x_pareto[mask]
y_fit    = y_pareto[mask]

popt, _  = curve_fit(log_func, x_fit, y_fit, maxfev=10000)
a_fit, b_fit = popt

y_pred   = log_func(x_fit, *popt)
ss_res   = np.sum((y_fit - y_pred) ** 2)
ss_tot   = np.sum((y_fit - np.mean(y_fit)) ** 2)
r2       = 1 - ss_res / ss_tot

x_smooth = np.linspace(0.001, 1.0, 500)
y_smooth = log_func(x_smooth, *popt)

print(f"Log fit: y = {a_fit:.4f} * ln(ε) + {b_fit:.4f}")
print(f"R²      = {r2:.4f}")

In [ ]:
from scipy.optimize import curve_fit
import numpy as np

def log_func(x, a, b):
    return a * np.log(x + 1e-6) + b

# Use greedy Pareto points for fit — excludes zero escape point
x_pareto = pareto_front_greedy['escape_risk'].values
y_pareto = pareto_front_greedy['cost_saving_pct'].values

mask    = x_pareto > 0.001
x_fit   = x_pareto[mask]
y_fit   = y_pareto[mask]

popt, _ = curve_fit(log_func, x_fit, y_fit, maxfev=10000)
a_fit, b_fit = popt

y_pred  = log_func(x_fit, *popt)
ss_res  = np.sum((y_fit - y_pred) ** 2)
ss_tot  = np.sum((y_fit - np.mean(y_fit)) ** 2)
r2      = 1 - ss_res / ss_tot

x_smooth = np.linspace(0.001, 1.0, 500)
y_smooth = log_func(x_smooth, *popt)

print(f"Log fit: y = {a_fit:.4f} * ln(ε) + {b_fit:.4f}")
print(f"R²      = {r2:.4f}")

# ── Figure 1 ─────────────────────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(7, 5))

ax1.scatter(
    all_points_df['escape_risk'],
    all_points_df['cost_saving_pct'],
    c='steelblue', alpha=0.25, s=15,
    label='Candidate subsets', zorder=1
)
ax1.plot(
    pareto_front_greedy['escape_risk'],
    pareto_front_greedy['cost_saving_pct'],
    'o-', color='black', linewidth=2.5,
    markersize=6, zorder=3,
    label='Approx. Pareto Front'
)
ax1.plot(
    x_smooth, y_smooth,
    color='gray', linewidth=2,
    linestyle='--', zorder=4,
    label='Log fit'
)
ax1.axvline(
    x=delta, color='red',
    linestyle='--', linewidth=2, zorder=5,
    label='Quality threshold'
)
ax1.scatter(
    [operating_point['escape_risk']],
    [operating_point['cost_saving_pct']],
    color='red', s=200, zorder=6, marker='*'
)
ax1.set_xlabel('Escape Risk (ε)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Cost Saving (%)', fontsize=14, fontweight='bold')
ax1.set_xlim(-0.02, 1.02)
ax1.set_ylim(-5, 105)
ax1.legend(fontsize=9, loc='lower right')
style_ax(ax1)
plt.tight_layout()
plt.savefig('fct_pareto_full.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: fct_pareto_full.png")

In [ ]:
fig2, ax2 = plt.subplots(figsize=(7, 5))

pf_zoom = pareto_front_greedy[
    pareto_front_greedy['escape_risk'] <= 0.004
].copy()

ax2.scatter(
    all_points_df['escape_risk'],
    all_points_df['cost_saving_pct'],
    c='steelblue', alpha=0.25, s=15,
    label='Candidate subsets', zorder=1
)
ax2.plot(
    pf_zoom['escape_risk'],
    pf_zoom['cost_saving_pct'],
    'o-', color='black', linewidth=2.5,
    markersize=6, zorder=3,
    label='Approx. Pareto Front'
)
ax2.axvline(
    x=delta, color='red',
    linestyle='--', linewidth=2, zorder=5,
    label='Quality threshold'
)
ax2.scatter(
    [operating_point['escape_risk']],
    [operating_point['cost_saving_pct']],
    color='red', s=200, zorder=6, marker='*'
)
ax2.set_xlabel('Escape Risk (ε)', fontsize=14, fontweight='bold')
ax2.set_ylabel('Cost Saving (%)', fontsize=14, fontweight='bold')
ax2.set_xlim(-0.0002, 0.004)
ax2.set_ylim(-1, 25)
ax2.legend(fontsize=9, loc='upper right')
style_ax(ax2)
plt.tight_layout()
plt.savefig('fct_pareto_zoom.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: fct_pareto_zoom.png")

In [ ]:
pareto_save = pareto_df.drop(columns=['selected_steps'])
pareto_save.to_csv("fct_rq2_pareto.csv", index=False)

pd.DataFrame({
    'test_id': operating_point['selected_steps']
}).to_csv("fct_rq2_cred.csv", index=False)

pd.DataFrame([{
    'operating_escaped':    operating_point['escaped_units'],
    'operating_steps':      int(operating_point['steps_selected']),
    'operating_cost_s':     operating_point['subset_cost_s'],
    'operating_saving_pct': operating_point['cost_saving_pct'],
    'operating_saving_s':   operating_point['cost_saving_s'],
    'c_full_cost_s':        round(C_full_cost, 2),
    'n_failing_units':      N_fail,
    'constraint_satisfied': True
}]).to_csv("fct_rq2_summary.csv", index=False)

print("Saved:")
print("  fct_rq2_pareto.csv")
print("  fct_rq2_cred.csv")
print("  fct_rq2_summary.csv")
print(f"\nOperating point:")
print(f"  Steps in Cred:  {int(operating_point['steps_selected'])}")
print(f"  Cost saving:    {operating_point['cost_saving_pct']:.2f}%")
print(f"  Zero escapes:   YES")